### Connexion au blob storage

In [0]:
# Connexion au blob storage
spark.conf.set("fs.azure.account.key.blobstoragehandson.blob.core.windows.net", "x")

### Import des parquets dans un dataframe

###### Requests

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType
from pyspark.sql import functions as F
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Paramètres
container = "next-best-offer"
storage_account = "blobstoragehandson"
sub_folder = "raw_data/requests/*/requests.parquet"

endpoint = f"wasbs://{container}@{storage_account}.blob.core.windows.net/{sub_folder}"

# Définition du schéma
requests_schema = StructType([
    StructField("CUST_NUM", LongType(), True),
    StructField("EVT_DT", TimestampType(), True)
])

# Lire tous les fichiers parquet
df_requests = spark.read.schema(requests_schema).parquet(endpoint)

# Identifier la date max réelle
max_date = df_requests.agg(F.max("EVT_DT")).collect()[0][0]
print("Dernière date disponible :", max_date)

# Déterminer le mois M-1 (dernier mois complet)
last_month_date = datetime(max_date.year, max_date.month, 1) - relativedelta(months=1)
year_month_target = (last_month_date.year, last_month_date.month)

print("Mois sélectionné (M-1) :", year_month_target)

# Ajouter colonnes année et mois TEMPORAIRES pour le filtrage
df_requests = df_requests.withColumn("_year_tmp", F.year("EVT_DT")) \
                         .withColumn("_month_tmp", F.month("EVT_DT"))

# Filtrer pour garder uniquement M-1
df_requests = df_requests.filter(
    (F.col("_year_tmp") == year_month_target[0]) &
    (F.col("_month_tmp") == year_month_target[1])
)

# Supprimer les colonnes temporaires
df_requests = df_requests.drop("_year_tmp", "_month_tmp")

In [0]:
df_requests.dropDuplicates()

### Feature Engineering

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import to_date, month, year

def preprocess_df(df: DataFrame, duration_col: str = None) -> DataFrame:
    """
    Standardise un DataFrame pour ton pipeline ML :
    - Convertit la colonne EVT_DT en Date
    - Crée les colonnes MONTH et YEAR
    - Renomme la colonne DURATION si spécifiée
    - Supprime systématiquement EVT_DT après création des colonnes MONTH/YEAR

    Args:
        df (DataFrame): DataFrame Spark
        duration_col (str, optional): nom pour renommer la colonne DURATION

    Returns:
        DataFrame: DataFrame traité
    """
    # Convertir EVT_DT en date
    df = df.withColumn("EVT_DT", to_date("EVT_DT"))
    
    # Renommer la colonne DURATION si nécessaire
    if duration_col:
        df = df.withColumnRenamed("DURATION", duration_col)
    
    # Créer MONTH et YEAR
    df = df.withColumn("MONTH", month("EVT_DT")) \
           .withColumn("YEAR", year("EVT_DT")) #\
           #.drop("EVT_DT")
    
    return df

In [0]:
df_requests = preprocess_df(df_requests)

##### Requests

In [0]:
df_requests = df_requests.drop("EVT_DT")

### Conversion du dataframe en delta table + test

In [0]:
%sql
DROP TABLE IF EXISTS nboRequestsAggregateM_1;

In [0]:
# Sauver le dataframe au format delta table natif de Databricks
df_requests.write.format("delta").saveAsTable("nboRequestsAggregateM_1")

In [0]:
%sql
-- Test ouverture en SQL de la delta table
SELECT * 
FROM nboRequestsAggregateM_1;